# TMDB Movie Data Analysis

This notebook fetches movie data from the TMDB API, cleans it, calculates KPIs and visualises the results.
The reusable code lives in `src/tmdb_analysis/`; this notebook runs it step by step and explains the results.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/, so go up one folder to find src/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from tmdb_analysis import config
from tmdb_analysis.api_client import load_movies
from tmdb_analysis.cleaning import (
    drop_irrelevant_columns,
    flatten_json_columns,
    convert_dtypes,
    replace_zeros_with_nan,
    convert_to_musd,
    convert_to_musd,
    fix_zero_votes,
    replace_text_placeholders,
    fix_zero_votes,
    replace_text_placeholders,
    remove_bad_rows,
    finalize_columns,
    FINAL_COLUMNS,

)

pd.set_option("display.max_columns", 30)

In [2]:
from tmdb_analysis.kpis import add_profit_and_roi, get_all_kpis
from tmdb_analysis.analysis import (
    search_bruce_willis_scifi_action,
    search_uma_thurman_tarantino,
    add_franchise_status,
    compare_franchise_vs_standalone,
    compare_franchise_vs_standalone,
    get_top_franchises,
    get_top_directors,

)

## Phase 1: Data Extraction

Movies are fetched from `/movie/{id}?append_to_response=credits`, so details and cast/crew come back in one request.
Each request has a 10-second timeout and is retried up to 3 times on rate limits or server errors.
ID `0` from the brief doesn't exist, so it is skipped. Six extra IDs are added because none of the lab
movies match the two required searches. The raw data is saved to `data/raw/`, so later runs don't call the API again.

In [3]:
df = load_movies()   # use load_movies(refresh=True) to call the API again

print(f"{len(df)} movies, {df.shape[1]} columns")
df.head(2)

Loading saved data from C:\Users\KING\Documents\AmaliTech\Specialization\pandas_numpys\TMDB-Movie-Data-Analysis\data\raw\movies_raw.json
24 movies, 28 columns


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,Avengers: Endgame,After the devastating events of Avengers: Infi...,82.3789,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,"[{'id': 420, 'logo_path': '/hUzeosd33nzE5MCNsZ...","[{'iso_3166_1': 'US', 'name': 'United States o...",2019-04-24,2938567784,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.247,29084,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,Avatar,"In the 22nd century, a paraplegic Marine is di...",37.2664,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,"[{'id': 444, 'logo_path': None, 'name': 'Dune ...","[{'iso_3166_1': 'US', 'name': 'United States o...",2009-12-16,2923706026,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.611,34892,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


## Phase 2: Data Cleaning

### 2.1 Drop irrelevant columns and flatten the JSON columns

Nested columns like `genres` become text such as `Action|Adventure`. Empty values become `NaN` (not `""`)
so they count as missing later. The `credits` column is split into `cast`, `cast_size`, `director` and `crew_size`.

In [4]:
df = drop_irrelevant_columns(df)
df = flatten_json_columns(df)

df[["title", "genres", "belongs_to_collection", "cast_size", "director", "crew_size"]].head()

,title,genres,belongs_to_collection,cast_size,director,crew_size
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,103,Anthony Russo|Joe Russo,613
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,68,James Cameron,999
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,99,J.J. Abrams,270
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,69,Anthony Russo|Joe Russo,738
4,Titanic,Drama|Romance,NaN,116,James Cameron,271


### 2.2 Inspect the extracted columns

Multi-value columns are split first, so each genre, language or company is counted on its own.

In [5]:
for column in ["genres", "spoken_languages", "production_countries", "production_companies"]:
    print(df[column].str.split("|").explode().value_counts().head(), "\n")

print(df["belongs_to_collection"].value_counts().head())

genres
Adventure          17
Action             15
Science Fiction    13
Thriller            6
Family              5
Name: count, dtype: int64 

spoken_languages
English     24
Español      4
Français     4
Pусский      4
             3
Name: count, dtype: int64 

production_countries
United States of America    23
United Kingdom               2
France                       1
Name: count, dtype: int64 

production_companies
Marvel Studios              5
Universal Pictures          3
Miramax                     3
A Band Apart                3
Lightstorm Entertainment    2
Name: count, dtype: int64 

belongs_to_collection
The Avengers Collection     4
Star Wars Collection        2
Jurassic Park Collection    2
Frozen Collection           2
Kill Bill Collection        2
Name: count, dtype: int64


## Key Details

Most Appeared Genre: Adventure~17,

Company Dominant ~ 5


### 2.3 Fix data types, replace unrealistic zeros, convert to millions

A budget, revenue or runtime of 0 means the value is unknown, so it becomes `NaN`.
This must happen **before** converting to millions; otherwise an unknown budget becomes `0.0`,
which makes ROI infinite and profit equal to the whole revenue.

In [6]:
print("Zero budgets before cleaning:", (df["budget"] == 0).sum())

df = convert_dtypes(df)
df = replace_zeros_with_nan(df)
df = convert_to_musd(df)

print("Zero values left in budget_musd:", (df["budget_musd"] == 0).sum())
print("Unknown budgets (NaN):", df["budget_musd"].isna().sum())
df[["title", "release_date", "budget_musd", "revenue_musd", "runtime"]].head()

Zero budgets before cleaning: 0
Zero values left in budget_musd: 0
Unknown budgets (NaN): 0


,title,release_date,budget_musd,revenue_musd,runtime
0,Avengers: Endgame,2019-04-24,356.0,2938.567784,181
1,Avatar,2009-12-16,237.0,2923.706026,162
2,Star Wars: The Force Awakens,2015-12-15,245.0,2068.223624,136
3,Avengers: Infinity War,2018-04-25,300.0,2052.415039,149
4,Titanic,1997-12-18,200.0,2264.162353,194


### 2.4 Zero votes and placeholder text

A `vote_average` from zero votes is not a real rating, so it becomes `NaN`; this keeps unrated movies
out of the rating rankings. Placeholder text such as "No Data" in `overview` and `tagline` also becomes `NaN`,
so missing text is counted as missing.

In [7]:
print("Movies with 0 votes:", (df["vote_count"] == 0).sum())

df = fix_zero_votes(df)
df = replace_text_placeholders(df)

print("Missing overview:", df["overview"].isna().sum())
print("Missing tagline:", df["tagline"].isna().sum())

Movies with 0 votes: 0
Missing overview: 0
Missing tagline: 0


### 2.5 Remove bad rows and finalise the columns

Duplicates, rows without an `id` or `title`, rows with fewer than 10 non-missing values and unreleased
movies are removed. The columns are then put in the order required by the brief.

In [8]:
print("Rows before:", len(df))

df = remove_bad_rows(df)
df = finalize_columns(df)

print("Rows after:", len(df))
df.head()

Rows before: 24
Rows after: 24


,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,production_countries,vote_count,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2938.567784,Marvel Studios,United States of America,29084,8.247,82.3789,181,After the devastating events of Avengers: Infi...,English|日本語|,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,103,Anthony Russo|Joe Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,United States of America|United Kingdom,34892,7.611,37.2664,162,"In the 22nd century, a paraplegic Marine is di...",English|Español,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,68,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure|Action|Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd.|Bad Robot,United States of America,21048,7.240,27.2795,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,Daisy Ridley|John Boyega|Oscar Isaac|Adam Driv...,99,J.J. Abrams,270
3,299536,Avengers: Infinity War,An entire universe. Once and for all.,2018-04-25,Adventure|Action|Science Fiction,The Avengers Collection,en,300.0,2052.415039,Marvel Studios,United States of America,33223,8.245,75.9928,149,As the Avengers and their allies have continue...,English|,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...,69,Anthony Russo|Joe Russo,738
4,597,Titanic,Nothing on earth could come between them.,1997-12-18,Drama|Romance,NaN,en,200.0,2264.162353,Paramount Pictures|20th Century Fox|Lightstorm...,United States of America,27948,7.901,42.1268,194,101-year-old Rose DeWitt Bukater tells the sto...,English|Français|Deutsch|svenska|Italiano|Pусский,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...,116,James Cameron,271


### 2.6 Check the cleaning

These checks prove the cleaning rules hold on the final data. Every value should be `True`.

In [9]:
checks = {
    "Columns in the required order": list(df.columns) == FINAL_COLUMNS,
    "No duplicate ids": df["id"].is_unique,
    "No zeros in budget_musd": (df["budget_musd"] != 0).all(),
    "No zeros in revenue_musd": (df["revenue_musd"] != 0).all(),
    "No rating without votes": df.loc[df["vote_count"] == 0, "vote_average"].isna().all(),
    "Every row has at least 10 values": (df.notna().sum(axis=1) >= 10).all(),
}
checks

{'Columns in the required order': True,
 'No duplicate ids': True,
 'No zeros in budget_musd': np.True_,
 'No zeros in revenue_musd': np.True_,
 'No rating without votes': np.True_,
 'Every row has at least 10 values': np.True_}

### 2.7 Save the clean data

In [10]:
config.CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(config.CLEAN_DATA_PATH, index=False)
print("Saved to", config.CLEAN_DATA_PATH)

Saved to C:\Users\KING\Documents\AmaliTech\Specialization\pandas_numpys\TMDB-Movie-Data-Analysis\data\processed\tmdb_movies_clean.csv


## Phase 3: KPI Analysis

### 3.1 Best and worst performing movies

Profit is revenue minus budget; ROI is revenue divided by budget. All 10 rankings from the brief use one
function, `rank_movies()`. ROI rankings only include movies with a budget of at least $10M, and rating rankings
only include movies with at least 10 votes, so a cheap or barely-rated movie can't top the list by chance.

In [11]:
df = add_profit_and_roi(df)

kpis = get_all_kpis(df)

for name, table in kpis.items():
    print(name)
    display(table)

Highest Revenue


,title,revenue_musd,budget_musd,vote_count
1,Avengers: Endgame,2938.567784,356.0,29084
2,Avatar,2923.706026,237.0,34892
3,Titanic,2264.162353,200.0,27948
4,Star Wars: The Force Awakens,2068.223624,245.0,21048
5,Avengers: Infinity War,2052.415039,300.0,33223


Highest Budget


,title,budget_musd,vote_count
1,Avengers: Endgame,356.0,29084
2,Avengers: Infinity War,300.0,33223
3,Star Wars: The Last Jedi,300.0,16800
4,The Lion King,260.0,10980
5,Star Wars: The Force Awakens,245.0,21048


Highest Profit


...

,title,vote_average,budget_musd,vote_count
1,Pulp Fiction,8.481,8.0,31149
2,Avengers: Endgame,8.247,356.0,29084
3,Avengers: Infinity War,8.245,300.0,33223
4,The Avengers,8.082,220.0,40123
5,Harry Potter and the Deathly Hallows: Part 2,8.081,125.0,22614


Lowest Rated (>= 10 votes)


,title,vote_average,budget_musd,vote_count
1,Jurassic World: Fallen Kingdom,6.520,170.0,13105
2,Jurassic World,6.703,150.0,22044
3,Star Wars: The Last Jedi,6.732,300.0,16800
4,Armageddon,6.835,140.0,8922
5,Looper,6.918,30.0,11293


Most Popular


,title,popularity,budget_musd,vote_count
1,Avengers: Endgame,82.3789,356.0,29084
2,Avengers: Infinity War,75.9928,300.0,33223
3,The Avengers,54.2058,220.0,40123
4,Harry Potter and the Deathly Hallows: Part 2,52.8117,125.0,22614
5,Avengers: Age of Ultron,46.3190,235.0,25217


### 3.2 Advanced search

Searches match whole names in the pipe-separated columns (`"Action|Science Fiction"` is split into
`["Action", "Science Fiction"]`), so "Bruce Willis" can't accidentally match a longer name, and a movie
must have *both* genres to qualify.

In [12]:
print("Search 1: Science Fiction Action movies starring Bruce Willis (best rated first)")
display(search_bruce_willis_scifi_action(df))

print("Search 2: Uma Thurman movies directed by Quentin Tarantino (shortest first)")
display(search_uma_thurman_tarantino(df))

Search 1: Science Fiction Action movies starring Bruce Willis (best rated first)


,title,vote_average,genres
18,The Fifth Element,7.592,Science Fiction|Action|Adventure
20,Looper,6.918,Action|Thriller|Science Fiction
19,Armageddon,6.835,Action|Science Fiction|Adventure


Search 2: Uma Thurman movies directed by Quentin Tarantino (shortest first)


,title,runtime,director
22,Kill Bill: Vol. 1,111,Quentin Tarantino
23,Kill Bill: Vol. 2,136,Quentin Tarantino
21,Pulp Fiction,154,Quentin Tarantino


### 3.3 Franchise vs. standalone movies

A movie is a franchise movie if it belongs to a collection. The two groups are compared on mean revenue,
median ROI, mean budget, mean popularity and mean rating. Median is used for ROI because a single cheap hit
can pull the mean far above a typical movie.

In [13]:
df = add_franchise_status(df)

compare_franchise_vs_standalone(df)

,movies,mean_revenue_musd,median_roi,mean_budget_musd,mean_popularity,mean_rating
franchise_status,,,,,,
Franchise,18,1522.03,7.31,197.11,36.5,7.45
Standalone,6,768.16,5.93,104.67,27.0,7.45


## Key Details

Franchise has much more Revenue than Standlone movies  
Franchise has more Median than Standlone movies  

### 3.4 Most successful franchises and directors

Both tables are ranked by total revenue. For directors, a co-directed movie counts for each director.

In [14]:
print("Top franchises")
display(get_top_franchises(df).head(10))

print("Top directors")
display(get_top_directors(df).head(10))

Top franchises


,movies,total_budget_musd,mean_budget_musd,total_revenue_musd,mean_revenue_musd,mean_rating
belongs_to_collection,,,,,,
The Avengers Collection,4,1111.0,277.75,7915.20,1978.80,7.96
Star Wars Collection,2,545.0,272.50,3402.63,1701.32,6.99
Jurassic Park Collection,2,320.0,160.00,2982.01,1491.00,6.61
Avatar Collection,1,237.0,237.00,2923.71,2923.71,7.61
Frozen Collection,2,300.0,150.00,2727.90,1363.95,7.23
The Lion King (Reboot) Collection,1,260.0,260.00,1662.02,1662.02,7.09
The Fast and the Furious Collection,1,190.0,190.00,1515.40,1515.40,7.21
Black Panther Collection,1,200.0,200.00,1349.93,1349.93,7.36
Harry Potter Collection,1,125.0,125.00,1341.51,1341.51,8.08


Top directors


,movies,total_revenue_musd,mean_rating
director,,,
James Cameron,2,5187.87,7.76
Joe Russo,2,4990.98,8.25
Anthony Russo,2,4990.98,8.25
Joss Whedon,2,2924.22,7.68
Chris Buck,2,2727.90,7.23
Jennifer Lee,2,2727.90,7.23
J.J. Abrams,1,2068.22,7.24
Colin Trevorrow,1,1671.54,6.70
Jon Favreau,1,1662.02,7.09


## Key Detail

Top 3 Directors in Revenue also have top Rating